# Morphoné Model via ContentVec - Feature Extraction, Training & Inference Pipeline
This notebook handles repository setup, pre-trained model downloads, dataset preparation via interactive widgets, configuration patching, model training, and inference for the DDSP-SVC model using the ContentVec encoder.

## Cell 1 — Repository Setup and Dependencies
Checks if the workspace is ready, clones the official DDSP-SVC repository if missing, and installs all required dependencies from `requirements.txt`.

In [ ]:
import os

# Clone the repository inside the current working folder if not already present
if not os.path.exists("preprocess.py"):
    print("Files not found locally. Cloning DDSP-SVC repository...")
    !git clone https://github.com/yxlllc/DDSP-SVC.git .
else:
    print("Repository files already present.")

print("Checking and installing dependencies in progress")
!pip install -r requirements.txt

## Cell 2 — Download Pre-trained Models
Downloads and sets up the required pre-trained checkpoint weights (ContentVec, NSF-HiFiGAN, and RMVPE) via the Hugging Face Hub API.

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

os.makedirs("pretrain/contentvec", exist_ok=True)
os.makedirs("pretrain/nsf_hifigan", exist_ok=True)

print("Start Downloading Templates via Official Hugging Face API\n")

if not os.path.exists("pretrain/contentvec/checkpoint_best_legacy_500.pt"):
    print("ContentVec is downloading")
    try:
        temp_path = hf_hub_download(repo_id="lj1995/VoiceConversionWebUI", filename="checkpoint_best_legacy_500.pt")
        shutil.move(temp_path, "pretrain/contentvec/checkpoint_best_legacy_500.pt")
        print("-> ContentVec downloaded and saved successfully.\n")
    except Exception as e:
        print(f"Error downloading ContentVec: {e}\n")
else:
    print("ContentVec is already on your local repository. Skip the download.\n")

if not os.path.exists("pretrain/nsf_hifigan/model") or not os.path.exists("pretrain/nsf_hifigan/config.json"):
    print("Downloading of NSF-HiFiGAN in progress")
    try:
        temp_model = hf_hub_download(repo_id="openvpi/models", filename="nsf_hifigan/model")
        shutil.move(temp_model, "pretrain/nsf_hifigan/model")

        temp_config = hf_hub_download(repo_id="openvpi/models", filename="nsf_hifigan/config.json")
        shutil.move(temp_config, "pretrain/nsf_hifigan/config.json")
        print("->NSF-HiFiGAN downloaded and saved successfully.\n")
    except Exception as e:
        print(f"Error downloading NSF-HiFiGAN: {e}\n")
else:
    print("Vocoder NSF-HiFiGAN is already on your local repository. Skip the download..\n")

if not os.path.exists("pretrain/rmvpe.pt"):
    print("Downloading of di RMVPE in progress")
    try:
        temp_rmvpe = hf_hub_download(repo_id="lj1995/VoiceConversionWebUI", filename="rmvpe.pt")
        shutil.move(temp_rmvpe, "pretrain/rmvpe.pt")
        print("-> RMVPE downloaded and saved successfully.\n")
    except Exception as e:
        print(f"Error downloading RMVPE: {e}\n")
else:
    print("RMVPE is already on your local repository. Skip the download.\n")

print("Model checking and downloading process completed.")

## Cell 3 — Local SSD Reset & Variables
Cleans up any existing temporary dataset folders on the local SSD and re-initializes global parameters (such as target instruments and sample limits).

In [ ]:
# Completely remove local temporary folder on SSD
LOCAL_SSD_BASE = "/content/dataset_temp"
if os.path.exists(LOCAL_SSD_BASE):
    shutil.rmtree(LOCAL_SSD_BASE)
    print("Local SSD folder successfully cleaned.")

# Reset global variables to the desired values ​​(e.g. brass acoustic and 2500 samples)
CHOSEN_INSTRUMENTS = ["brass_acoustic"]
MAX_SAMPLES = 2500

print(f"Configuration reset and reset:")
print(f"Instrument: {CHOSEN_INSTRUMENTS}")
print(f"Target samples: {MAX_SAMPLES}")

## Cell 4 — Interactive Instrument Configuration
Displays interactive toggle buttons and sliders allowing you to select up to 5 acoustic instruments and define the target training sample size.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

instruments_avaiable = [
    "brass_acoustic",
    "flute_acoustic",
    "guitar_acoustic",
    "keyboard_acoustic",
    "mallet_acoustic",
    "organ_acoustic",
    "reed_acoustic",
    "string_acoustic",
    "vocal_acoustic"
]

print("Configuration of Acoustic Instruments")
print(f"Select up to 5 instruments and set samples.\n")

# Interactive buttons
checkboxes = [
    widgets.ToggleButton(
        value=False,
        description=instrument,
        disabled=False,
        button_style='',
        icon='check'
    ) for instrument in instruments_avaiable
]

def on_button_change(change):
    active = [b for b in checkboxes if b.value == True]
    if len(active) > 5:
        change['owner'].value = False
        print("Warning: You can select a maximum of 5 instruments at the same time.")
    else:
        for b in checkboxes:
            b.button_style = 'success' if b.value else ''

for b in checkboxes:
    b.observe(on_button_change, names='value')

grid_instruments = widgets.GridBox(
    checkboxes,
    layout=widgets.Layout(grid_template_columns='repeat(3, 180px)', grid_gap='8px')
)

# Slider for samples
slider_samples = widgets.IntSlider(
    value=2500,
    min=1,
    max=5000,
    step=1,
    description='Samples:',
    disabled=False,
    continuous_update=False
)

# Confirm button
btn_confirm = widgets.Button(
    description='Confirm Settings',
    disabled=False,
    button_style='primary',
    icon='save'
)

CHOSEN_INSTRUMENTS = []
MAX_SAMPLES = 2500
out = widgets.Output()

def on_confirm_clicked(b):
    global CHOSEN_INSTRUMENTS, MAX_SAMPLES
    with out:
        clear_output()
        CHOSEN_INSTRUMENTS = [box.description for box in checkboxes if box.value == True]
        MAX_SAMPLES = slider_samples.value

        if not CHOSEN_INSTRUMENTS:
            print("Error: You must select at least one instrument")
        else:
            print("Configuration saved successfully")
            print(f"Selected acoustic instruments ({len(CHOSEN_INSTRUMENTS)}/5): {CHOSEN_INSTRUMENTS}")
            print(f"Number of samples per instrument: {MAX_SAMPLES}")

btn_confirm.on_click(on_confirm_clicked)

display(widgets.VBox([
    widgets.Label("Choose acoustic instruments (max 5):"),
    grid_instruments,
    widgets.HTML("<hr>"),
    widgets.Label("Choose the maximum number of samples for training:"),
    slider_samples,
    widgets.HTML("<br>"),
    btn_confirm,
    out
]))

=== CONFIGURAZIONE STRUMENTI (NSYNTH) ===
Seleziona fino a un massimo di 5 strumenti e imposta i campioni.



## Cell 5 — Dataset Copy to Local SSD
Scans the repository audio folders, matches files for the chosen instruments, and uses multi-threading to quickly copy them to the local SSD for training.

In [ ]:
import random
import concurrent.futures
from tqdm.notebook import tqdm

try:
    if 'CHOSEN_INSTRUMENTS' in globals() and CHOSEN_INSTRUMENTS:
         active_instruments = CHOSEN_INSTRUMENTS
    else:
        active_instruments = ["brass_acoustic"]

    if 'MAX_SAMPLES' in globals():
        max_active_samples = int(MAX_SAMPLES)
    else:
        max_active_samples = 2500
except Exception:
    active_instruments = ["brass_acoustic"]
    max_active_samples = 2500

CHOSEN_INSTRUMENTS = active_instruments
MAX_SAMPLES = max_active_samples

print(f"Copy of dataset subset to Colab Local SSD DISKB")
print(f"Syncronized -> Intruments: {CHOSEN_INSTRUMENTS} | Target samples: {MAX_SAMPLES}\n")

DRIVE_TRAIN_AUDIO = "./data/train/audio"
DRIVE_VAL_AUDIO = "./data/val/audio"
LOCAL_SSD_BASE = "/content/dataset_temp"

if os.path.exists(LOCAL_SSD_BASE):
    !rm -rf {LOCAL_SSD_BASE}
os.makedirs(LOCAL_SSD_BASE, exist_ok=True)

def completed_copy(src_dir, dest_dir, is_train=True):
    limite = MAX_SAMPLES if is_train else 10
    folder_dest = os.path.join(dest_dir, "audio")
    os.makedirs(folder_dest, exist_ok=True)

    print(f"Scan for the instrument {CHOSEN_INSTRUMENTS} ({'TRAIN' if is_train else 'VAL'})...")

    couples = []
    if os.path.exists(src_dir):
        with os.scandir(src_dir) as entries:
            for entry in entries:
                if len(couples) >= limite:
                    break
                if entry.is_file():
                    name = entry.name
                    if any(name.startswith(s + "_") for s in CHOSEN_INSTRUMENTS) and name.endswith('.wav'):
                        src_path = entry.path
                        dst_path = os.path.join(folder_dest, name)
                        couples.append((src_path, dst_path))

    def cpy_attempt(path):
        src, dst = path
        try:
            shutil.copy(src, dst)
            return True
        except Exception:
            return False

    desc_label = f"Copy on SSD ({'TRAIN' if is_train else 'VAL'})"

    if couples:
        with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
            results = list(tqdm(executor.map(cpy_attempt, couples), total=len(couples), desc=desc_label))
        total_copied = sum(1 for r in results if r)
        print(f"Successfully copied {total_copied} files of {CHOSEN_INSTRUMENTS} on {limite} requested.\n")
    else:
        print(f"Nonen file found for {CHOSEN_INSTRUMENTS} inside {src_dir}\n")

completed_copy(DRIVE_TRAIN_AUDIO, os.path.join(LOCAL_SSD_BASE, "train"), is_train=True)
completed_copy(DRIVE_VAL_AUDIO, os.path.join(LOCAL_SSD_BASE, "validation"), is_train=False)

print("LOCAL COPY COMPLETED")

## Cell 6 — YAML Config & Preprocessing Pipeline
Updates the YAML configuration file to point to the local SSD paths and runs feature extraction preprocessing.

In [ ]:
import yaml

config_path = 'configs/reflow.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config['data']['train_path'] = '/content/dataset_temp/train'
config['data']['valid_path'] = '/content/dataset_temp/validation'

if 'CHOSEN_INSTRUMENTS' in globals():
    config['model']['n_spk'] = len(CHOSEN_INSTRUMENTS)
else:
    config['model']['n_spk'] = 1

if 'train' in config:
    config['train']['cache_all_data'] = False

with open(config_path, 'w') as f:
    yaml.safe_dump(config, f)

print("\nFile configs/reflow.yaml configured correctly to run locally on SSD.")

# Run preprocessing for feature extraction (F0 and encoder features) using 2 parallel workers
!python preprocess.py -c configs/reflow.yaml -j 2

## Cell 7 — Training Pipeline
Applies necessary codebase patches and starts the model training loop.

In [ ]:
# Copy pitch_aug_dict.npy from validation to train
src_dict = '/content/dataset_temp/validation/pitch_aug_dict.npy'
dst_dict = '/content/dataset_temp/train/pitch_aug_dict.npy'

if os.path.exists(src_dict):
    shutil.copy(src_dict, dst_dict)
    print("File pitch_aug_dict.npy successfully copied to train")
else:
    print("File not found. Please preprocess first..")

# Relative local path to the data_loaders.py file
file_loader = "reflow/data_loaders.py"

with open(file_loader, "r") as f:
    code = f.read()

old_line = "aug_shift = self.pitch_aug_dict[name_ext]"
new_line = "aug_shift = self.pitch_aug_dict.get(name_ext, 0)"

if old_line in code:
    code = code.replace(old_line, new_line)
    with open(file_loader, "w") as f:
        f.write(code)
    print("Patch successfully applied to data_loaders.py")
else:
    print("Patch already applied or file structure changed")

# Start the model training process using the specified configuration file
!python train_reflow.py -c configs/reflow.yaml

## Cell 8 — Morphing and Inference UI
Provides an interactive widget interface to load trained model checkpoints, select source audio files, adjust pitch shifting parameters, and execute the final voice/instrument conversion.


In [ ]:
import subprocess
from IPython.display import Audio

style = {'description_width': 'initial'}

input_folder_w = widgets.Text(
    value='./data/test',
    description='Source Folder:',
    style=style,
    layout=widgets.Layout(width='80%')
)

input_file_w = widgets.Dropdown(
    options=[f for f in os.listdir(input_folder_w.value) if f.endswith('.wav')] if os.path.exists(input_folder_w.value) else [],
    description='Audio File:',
    style=style,
    layout=widgets.Layout(width='80%')
)

update_file_btn = widgets.Button(
    description='Update File',
    button_style='info',
    layout=widgets.Layout(width='15%')
)

# 2. Checkpoint selection
exp_path = './exp'
model_w = widgets.Dropdown(
    options=[os.path.join(r, f) for r, _, files in os.walk(exp_path) for f in files if f.endswith('.pt')] if os.path.exists(exp_path) else [],
    description='Select Checkpoint:',
    style=style,
    layout=widgets.Layout(width='80%')
)

update_model_btn = widgets.Button(
    description='Update Models',
    button_style='info',
    layout=widgets.Layout(width='15%')
)

# 3. Slider octaves/semitones
octave_w = widgets.IntSlider(
    value=0,
    min=-24,
    max=24,
    step=1,
    description='Shifting (Semitones):',
    style=style,
    layout=widgets.Layout(width='80%')
)

# 4. ID Speaker
spk_id_w = widgets.IntText(
    value=1,
    description='ID Instrument (Speaker ID):',
    style=style,
    layout=widgets.Layout(width='30%')
)

# 5. Output file name
output_name_w = widgets.Text(
    value='Instrument_output.wav',
    description='Output File Name:',
    style=style,
    layout=widgets.Layout(width='80%')
)

output_folder_w = widgets.Text(
    value='./outputs',
    description='Save in (Local Folder):',
    style=style,
    layout=widgets.Layout(width='80%')
)

start_btn = widgets.Button(
    description='Starts Morphing',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='30%', height='40px')
)

output_log = widgets.Output()

def on_update_file_clicked(b):
    if os.path.exists(input_folder_w.value):
        input_folder_w.options = [f for f in os.listdir(input_folder_w.value) if f.endswith('.wav')]

def on_update_model_clicked(b):
    if os.path.exists(exp_path):
        model_w.options = [os.path.join(r, f) for r, _, files in os.walk(exp_path) for f in files if f.endswith('.pt')]

def on_start_clicked(b):
    with output_log:
        clear_output()
        source_file = os.path.join(input_folder_w.value, input_file_w.value) if input_file_w.value else ""
        model_file = model_w.value
        destination_folder = output_folder_w.value
        destination_folder = os.path.join(destination_folder, output_name_w.value)

        if not source_file or not os.path.exists(source_file):
            print("Error: Please select a valid input audio file.")
            return
        if not model_file or not os.path.exists(model_file):
            print("Error: Please select a valid model checkpoint.")
            return

        os.makedirs(destination_folder, exist_ok=True)

        settings = [
            "python", "main_reflow.py",
            "-i", source_file,
            "-m", model_file,
            "-o", destination_file,
            "-k", str(octave_w.value),
            "-id", str(spk_id_w.value)
        ]

        print(f"Source file: {source_file}")
        print(f"Model: {model_file}")
        print(f"Saving inside: {destination_file}\n")
        print("Processing in progress...")

        process = subprocess.run(settings, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        if process.returncode == 0:
            print("Morphing completed")
            display(Audio(destination_file))
        else:
            print("Error during morphing process:")
            print(process.stderr)

update_file_btn.on_click(on_update_file_clicked)
update_model_btn.on_click(on_update_model_clicked)
start_btn.on_click(on_start_clicked)

display(widgets.HBox([input_folder_w, update_file_btn]))
display(input_file_w)
display(widgets.HBox([model_w, update_model_btn]))
display(octave_w)
display(spk_id_w)
display(output_name_w)
display(output_folder_w)
print("")
display(start_btn)
display(output_log)